### Supress Warnings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_2 = pd.read_csv(
    'coffee_shop_transactions_cleaned.csv',
    parse_dates = ['DateTime']   # parse_dates → converts the column to datetime64 on load
                                 # without this, DateTime loads as plain text (string)
                                 # and .dt.date / .dt.hour will throw AttributeError
)

print('\n Number of Rows,Cols : ', df_2.shape)
print('\n DateTime dtype      :', df_2['DateTime'].dtype)  # should be datetime64[ns]

df_2

In [ ]:
from IPython.display import Image

In [ ]:
Image(url="Coffee-Shop-EDA-Charts-1.jpg")

In [ ]:
Image(url="Coffee-Shop-EDA-Charts-2.jpg")

***

## 📊 Section 3 — Exploratory Data Analysis (EDA)

| # | Analysis | Pandas / Seaborn / Matplotlib method used |
|---|---|---|
| 1 | Dataset Snapshot | `.describe()` `.nunique()` `.sum()` |
| 2 | Items Ordered | `value_counts()` → `sns.barplot` |
| 3 | Revenue by Item | `groupby` + `sum()` → `sns.barplot` (horizontal) |
| 4 | Payment Method Split | `value_counts()` → `axes[0].pie` + `sns.barplot` |
| 5 | Busiest Hours | `.dt.hour` → `sns.countplot` |
| 6 | Correlation Heatmap | `.corr()` → `sns.heatmap(annot=True)` |
| 7 | Key Insights Summary | `groupby` + `idxmax()` |


### Set Global style — applies to all charts below

In [ ]:
# sns.set_theme(style='whitegrid', palette='pastel', context='talk')
sns.set_theme(style='darkgrid', palette='deep', context='notebook')

#### 1️⃣ Dataset Snapshot
> Quick numbers before we plot anything.

In [ ]:
print(f'Rows              : {len(df_2)}')
print(f'Unique items      : {sorted(df_2["Item"].unique())}')
print(f'Payment methods   : {sorted(df_2["PaymentMethod"].unique())}')
print(f'Total revenue     : ${df_2["TotalPrice"].sum():.2f}')
print(f'Avg order value   : ${df_2["TotalPrice"].mean():.2f}')
print()
df_2[['PricePerItem','Quantity','TotalPrice']].describe().round(2)

#### 2️⃣ How Many Times Was Each Item Ordered?
> `value_counts()` counts rows per item.  
> `sns.barplot` draws it with one line.

#### We use a bar chart because we are comparing the number of orders for different items. The height of each bar makes it easy to see which item is the most or least popular.

In [ ]:
# value_counts() — counts how many times each item appears in the Item column
item_counts = df_2['Item'].value_counts().reset_index()
item_counts.columns = ['Item', 'Orders']   # rename columns for clarity

plt.figure(
    figsize=(9, 4)    # width=9 inches, height=4 inches
)

sns.barplot(
    data     = item_counts,  # DataFrame to use
    x        = 'Item',       # column for the x-axis (categories)
    y        = 'Orders',     # column for the y-axis (bar height)
    palette  = 'Blues_d'     # colour palette — dark-to-light blue gradient
)

plt.title('Number of Orders per Item')  # chart heading
plt.xticks(rotation=15)                 # tilt x-axis labels 15° so they don't overlap
plt.tight_layout()                      # auto-adjust spacing so nothing is clipped
plt.show()                              # render the chart

#### 3️⃣ Total Revenue by Item
> `groupby` + `sum()` then `sns.barplot` with `orient='h'` for a horizontal layout.

#### We use a horizontal bar chart because we want to rank items by revenue. The longer the bar, the more revenue that item generated.

In [ ]:
# groupby('Item') — group all rows that share the same item name
# ['TotalPrice'].sum() — add up TotalPrice within each group
# reset_index() — converts the result back into a normal DataFrame
rev = df_2.groupby('Item')['TotalPrice'].sum().reset_index()
rev = rev.sort_values('TotalPrice', ascending=False)  # highest revenue first

plt.figure(
    figsize=(9, 5)    # width=9 inches, height=5 inches
)

sns.barplot(
    data     = rev,          # DataFrame to use
    y        = 'Item',       # y-axis = categories → makes the bar horizontal
    x        = 'TotalPrice', # x-axis = bar length (revenue amount)
    palette  = 'Blues_d'     # dark-to-light blue gradient
)

plt.title('Total Revenue by Item')  # chart heading
plt.xlabel('Revenue ($)')           # label on the x-axis
plt.tight_layout()                  # prevent labels being cut off
plt.show()

#### 4️⃣ Payment Method Split
> Pie chart (matplotlib) for share of transactions.  
> `sns.barplot` for revenue per method.

#### We use a pie chart because we want to show how transactions are divided among payment methods. Each slice represents a percentage of the whole.

#### We use a bar chart because we want to compare revenue amounts. Bar lengths make it easier to see which payment method generated more money.

In [ ]:
pay_count   = df_2['PaymentMethod'].value_counts()                               # count transactions per method
pay_revenue = df_2.groupby('PaymentMethod')['TotalPrice'].sum().reset_index()    # total revenue per method

fig, axes = plt.subplots(
    1, 2,             # 1 row, 2 columns — two charts side by side
    figsize=(12, 4)   # total figure width=12, height=4
)

# ── Left chart: Pie ────────────────────────────────────────────────────
axes[0].pie(
    pay_count.values,                    # numeric values that define slice sizes
    labels      = pay_count.index,       # label each slice with the payment method name
    autopct     = '%1.1f%%',             # show percentage inside each slice (1 decimal)
    startangle  = 140,                   # rotate the first slice to 140° (looks balanced)
    colors      = sns.color_palette('pastel')  # soft pastel colours from seaborn
)
axes[0].set_title('Transaction Share by Payment Method')

# ── Right chart: Bar ───────────────────────────────────────────────────
sns.barplot(
    data     = pay_revenue,      # DataFrame with payment method + revenue
    x        = 'PaymentMethod',  # x-axis categories
    y        = 'TotalPrice',     # bar height = revenue
    palette  = 'pastel',         # soft colour palette
    ax       = axes[1]           # draw into the RIGHT subplot (index 1)
)
axes[1].set_title('Revenue by Payment Method')
axes[1].set_xlabel('Method')
axes[1].set_ylabel('Revenue ($)')

plt.tight_layout()
plt.show()

#### 5️⃣ Busiest Hours
> `.dt.hour` extracts the hour.  
> `sns.countplot` counts transactions per hour automatically — no manual groupby needed.

#### We use a count plot because we want to count how many transactions happened during each hour of the day.

In [ ]:
# .dt.hour — extracts the hour (0–23) from a DateTime column
df_2['Hour'] = df_2['DateTime'].dt.hour

plt.figure(
    figsize=(9, 4)   # width=9, height=4
)

sns.countplot(
    data     = df_2,     # full DataFrame — countplot counts rows automatically
    x        = 'Hour',   # count how many rows exist for each unique hour value
    palette  = 'Blues_d' # dark-to-light blue gradient per bar
    # NOTE: no need for groupby — countplot does the counting internally
)

plt.title('Transactions by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Transactions')
plt.tight_layout()
plt.show()

#### 6️⃣ Correlation Heatmap
> `.corr()` measures how strongly numeric columns move together.  
> `sns.heatmap` with `annot=True` prints the values inside each cell automatically.

#### We use a heatmap because colors make correlations easy to interpret. Strong positive relationships appear in one color, negative relationships in another, and values near zero indicate little or no relationship.

In [ ]:
# .corr() — computes pairwise correlation between numeric columns
# Values range from -1 (perfect negative) to +1 (perfect positive), 0 = no link
corr = df_2[['PricePerItem','Quantity','TotalPrice']].corr().round(2)
print(corr)

plt.figure(
    figsize=(6, 4)   # small square figure suits a 3×3 grid
)

sns.heatmap(
    corr,               # the correlation matrix (DataFrame)
    annot      = True,  # print the numeric value inside each cell
    fmt        = '.2f', # format the annotation to 2 decimal places
    cmap       = 'coolwarm',  # red = positive correlation, blue = negative
    linewidths = 0.5,   # thin white grid lines between cells
    vmin       = -1,    # anchor the colour scale minimum at -1
    vmax       = 1      # anchor the colour scale maximum at +1
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

#### 7️⃣ Key Insights Summary
> Pulls together the most important numbers discovered above.

In [ ]:
# idxmax() — returns the INDEX (label) of the row with the highest value
# Used here to find the name of the best item / top payment method / busiest hour

best_item  = df_2.groupby('Item')['TotalPrice'].sum().idxmax()              # item with highest total revenue
best_pay   = df_2.groupby('PaymentMethod')['TotalPrice'].sum().idxmax()     # payment method with highest revenue
busy_hour  = df_2.groupby('Hour')['TransactionID'].count().idxmax()         # hour with most transactions

print('=' * 50)
print('  ☕  COFFEE SHOP — EDA SUMMARY')
print('=' * 50)
print(f'  Total transactions : {len(df_2)}')
print(f'  Total revenue      : ${df_2["TotalPrice"].sum():.2f}')   # .sum() adds all TotalPrice values
print(f'  Avg order value    : ${df_2["TotalPrice"].mean():.2f}')  # .mean() divides sum by count
print(f'  Best-selling item  : {best_item}')
print(f'  Top payment method : {best_pay}')
print(f'  Busiest hour       : {busy_hour}:00')
print('=' * 50)

***

***